# DATASCAPE 2026 — SigLIP2-first data pipeline

The experiment order is fixed:

1. frozen SigLIP2 embeddings from four rotations;
2. embedding candidates → geometric verification → connected duplicate components;
3. rebuild all duplicate components, then split an unseen label-noisy holdout at component level and rebuild `StratifiedGroupKFold` on development rows;
4. compare one-seed ConvNeXt OOF with SigLIP2 linear-probe OOF repeated across grouped-CV seeds;
5. measure dedup delta, cross-model and cross-partition agreement, and new OOF noise matrices;
6. only then consider bias tuning, noise audit, and a two-stage model.

Deduplication changes grouping only. It never deletes a row or changes a label. No pHash is used. The holdout is unseen by model selection, but its original labels may still be noisy; it is written to disk and not evaluated by default.

In [ ]:
# Kaggle API bootstrap: intentionally hardcoded for one-click Colab execution.
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API configured for this runtime.')


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow', 'cv2': 'opencv-python',
    'pillow_avif': 'pillow-avif-plugin',
    'torch': 'torch', 'transformers': 'transformers>=4.49.0', 'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import hashlib
import json
import gc
import math
import platform
import random
import threading
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pillow_avif  # registers AVIF support in Pillow; one source file is AVIF mislabeled as .jpg
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor

IN_COLAB = 'google.colab' in sys.modules
print('Python:', sys.version.split()[0], '| torch:', torch.__version__, '| Colab:', IN_COLAB)


## 1. Configuration

The SigLIP2 revision is pinned. The fixed-resolution 384 checkpoint matches the downstream image scale. Adjust paths for Drive/Colab before running.

In [ ]:
PROJECT_ROOT = Path('/content') if IN_COLAB else Path.cwd()
COMPETITION = 'datascape-2026-data-mining-competition'
DATA_ROOT = PROJECT_ROOT / COMPETITION
SOURCE_MANIFEST = PROJECT_ROOT / 'artifacts/preprocessing_colab/training_manifest.csv'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/siglip2_first_pipeline')
else:
    OUTPUT_DIR = PROJECT_ROOT / 'artifacts/siglip2_first_pipeline'
GENERATED_SOURCE_MANIFEST = OUTPUT_DIR / 'source_training_manifest.csv'
REJECTED_IMAGE_REPORT = OUTPUT_DIR / 'unreadable_training_images.csv'
DOWNLOAD_EXTRACT_DIR = Path('/content/datascape_dataset')
LOCAL_DATA_ONLY = os.environ.get('DATASCAPE_LOCAL_ONLY', '0') == '1'
REQUIRE_RESUME_CACHE_IN_COLAB = True
RESUME_CACHE_CANDIDATES = (
    Path('/content/siglip2_resume_cache.zip'),
    Path('/content/drive/MyDrive/DATASCAPE/siglip2_resume_cache.zip'),
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'google/siglip2-base-patch16-384'
MODEL_REVISION = 'f775b65a79762255128c981547af89addcfe0f88'
GROUPING_PROTOCOL_VERSION = 'siglip2_geometry_25000_component_split_v2'
ROTATIONS = (0, 90, 180, 270)
EMBED_BATCH_SIZE = 24
K_NEIGHBOURS = 20
KNN_BLOCK_SIZE = 512
MAX_GEOMETRY_CANDIDATES = 25000
GEOMETRY_WORKERS = max(1, min(8, (os.cpu_count() or 4) - 2))
GEOMETRY_CHECKPOINT_EVERY = 250
GEOMETRY_VERIFIER_VERSION = 'denoise_sift12_homography060_corr070_v1'
CALIBRATION_SAMPLE_SIZE = 512
CALIBRATION_SEED = 2603
SYNTHETIC_RECALL_QUANTILE = 0.01
DIRECT_MERGE_MIN_PAIRS = 20
DIRECT_MERGE_MIN_GEOMETRY_RATE = 0.80

SIFT_MIN_MATCHES = 12
HOMOGRAPHY_MIN_INLIER = 0.60
ALIGNED_MIN_CORRELATION = 0.70

HOLDOUT_FRACTION_FOLDS = 10  # first SGKF split is the unseen ~10% holdout
HOLDOUT_SEED = 2604
HOLDOUT_NN_THRESHOLDS = (0.90, 0.925, 0.95, 0.96, 0.97, 0.98, 0.99)
HOLDOUT_NN_REVIEW_COSINE = 0.95
OOF_FOLDS = 5
FOLD_SEED = 2605
PROBE_FOLD_SEEDS = (2605, 2606, 2607)
PROBE_MODEL_SEED = 1701
PROBE_FEATURE_MODE = 'rot0_plus_mean'
LINEAR_PROBE_EPOCHS = 40
LINEAR_PROBE_BATCH_SIZE = 256

CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}

# Same ConvNeXt architecture/seed/hyperparameters are required for a dedup delta.
CONVNEXT_OLD_GROUPS_OOF = None
CONVNEXT_NEW_GROUPS_OOF = None
EVALUATE_HOLDOUT = False
PUBLIC_LEADERBOARD_SUBMISSION_BUDGET = 3  # baseline, final candidate, verified-bug contingency


## 2. Load the original rows

Every source row is retained. Existing `split_group` and pHash fields are ignored when new components are built; exact decoded-pixel hashes and verified SigLIP2 neighbours define the new grouping.

In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def install_resume_cache_if_available():
    required_cache_files = (
        'source_training_manifest.csv',
        'siglip2_four_rotation_embeddings.npy',
        'siglip2_mean_embeddings.npy',
        'embedding_threshold_calibration.json',
        'synthetic_duplicate_embedding_calibration.csv',
        'geometry_metrics_cache.csv',
        'stage5_summary.json',
    )
    if all((OUTPUT_DIR / name).is_file() for name in required_cache_files):
        print('Resume cache already exists in the persistent output directory.')
        return
    archive = next((path for path in RESUME_CACHE_CANDIDATES if path.is_file()), None)
    if archive is not None:
        print('Installing resume cache:', archive)
        safe_extract(archive, OUTPUT_DIR)
    missing = [name for name in required_cache_files if not (OUTPUT_DIR / name).is_file()]
    if missing and IN_COLAB and REQUIRE_RESUME_CACHE_IN_COLAB:
        raise RuntimeError(
            'Resume cache is required to prevent recomputing SigLIP2. Upload '
            'siglip2_resume_cache.zip to /content or MyDrive/DATASCAPE, then '
            'run this cell again. Missing: ' + ', '.join(missing)
        )


def find_dataset_root(start):
    start = Path(start).expanduser().resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Train_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(f'Dataset root not found under {start}')


def resolve_dataset_root():
    local_candidates = [DATA_ROOT, Path('/content') / COMPETITION, DOWNLOAD_EXTRACT_DIR]
    for candidate in local_candidates:
        if candidate.exists():
            try:
                return find_dataset_root(candidate)
            except FileNotFoundError:
                pass

    if LOCAL_DATA_ONLY:
        raise FileNotFoundError(
            f'Local-only mode is active, but the dataset was not found at {DATA_ROOT}. '
            'Kaggle download is disabled for the Mac runner.'
        )

    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(KAGGLE_API_TOKEN)
    print('Dataset not found locally; downloading Kaggle competition data...')
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, DOWNLOAD_EXTRACT_DIR))
    return find_dataset_root(downloaded)


def decoded_pixel_sha256(path):
    with Image.open(path) as opened:
        rgb = np.ascontiguousarray(np.asarray(ImageOps.exif_transpose(opened).convert('RGB')))
    digest = hashlib.sha256()
    digest.update(np.asarray(rgb.shape, dtype=np.int64).tobytes())
    digest.update(rgb.tobytes())
    return digest.hexdigest()


def list_training_images(dataset_root):
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
    image_paths = []
    for label in CLASSES:
        image_paths.extend(
            (label, path) for path in sorted((dataset_root / 'Train_Set' / label).rglob('*'))
            if path.is_file() and path.suffix.lower() in extensions
        )
    return image_paths


def manifest_matches_dataset(manifest_path, dataset_root, rejected_path):
    if not manifest_path.is_file():
        return False
    try:
        manifest = pd.read_csv(manifest_path, usecols=['filepath', 'label'])
    except (ValueError, pd.errors.ParserError):
        return False
    expected = {
        (path.relative_to(dataset_root).as_posix(), label)
        for label, path in list_training_images(dataset_root)
    }
    observed = set(zip(manifest.filepath.astype(str), manifest.label.astype(str)))
    rejected = set()
    if rejected_path.is_file():
        try:
            rejected_frame = pd.read_csv(rejected_path, usecols=['filepath', 'label'])
            rejected = set(zip(rejected_frame.filepath.astype(str), rejected_frame.label.astype(str)))
        except (ValueError, pd.errors.ParserError):
            return False
        for relative, _ in rejected:
            try:
                with Image.open(dataset_root / relative) as opened:
                    opened.load()
                return False
            except OSError:
                pass
    return (
        len(manifest) == len(observed)
        and observed.isdisjoint(rejected)
        and observed | rejected == expected
    )


def build_source_manifest(dataset_root, destination, rejected_destination):
    rows = []
    rejected_rows = []
    image_paths = list_training_images(dataset_root)
    for label, path in tqdm(image_paths, desc='building source manifest'):
        relative = path.relative_to(dataset_root).as_posix()
        try:
            pixel_sha256 = decoded_pixel_sha256(path)
        except OSError as error:
            rejected_rows.append({
                'filepath': relative,
                'filename': path.name,
                'label': label,
                'error': f'{type(error).__name__}: {error}',
            })
            continue
        rows.append({
            'filepath': relative,
            'filename': path.name,
            'label': label,
            'pixel_sha256': pixel_sha256,
            'source_path': str(path),
        })
    destination.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(destination, index=False)
    pd.DataFrame(
        rejected_rows,
        columns=['filepath', 'filename', 'label', 'error'],
    ).to_csv(rejected_destination, index=False)
    if rejected_rows:
        print(f'Skipped {len(rejected_rows)} unreadable training image(s): {rejected_destination}')
    return destination


install_resume_cache_if_available()
DATA_ROOT = resolve_dataset_root()
if not manifest_matches_dataset(SOURCE_MANIFEST, DATA_ROOT, REJECTED_IMAGE_REPORT):
    if manifest_matches_dataset(GENERATED_SOURCE_MANIFEST, DATA_ROOT, REJECTED_IMAGE_REPORT):
        SOURCE_MANIFEST = GENERATED_SOURCE_MANIFEST
        print('Using the complete cached manifest:', SOURCE_MANIFEST)
    else:
        if SOURCE_MANIFEST.is_file():
            print('Existing source manifest is incomplete or stale; rebuilding from the local Train_Set.')
        SOURCE_MANIFEST = build_source_manifest(
            DATA_ROOT,
            GENERATED_SOURCE_MANIFEST,
            REJECTED_IMAGE_REPORT,
        )
print('Dataset root:', DATA_ROOT)
print('Source manifest:', SOURCE_MANIFEST)


frame = pd.read_csv(SOURCE_MANIFEST).reset_index(drop=True)
required_columns = {'filepath', 'filename', 'label', 'pixel_sha256'}
missing = required_columns - set(frame.columns)
if missing:
    raise ValueError(f'Missing source-manifest columns: {sorted(missing)}')
if set(frame.label) != set(CLASSES):
    raise ValueError('Unexpected labels.')

def resolve_image_path(relative, source=''):
    candidates = [DATA_ROOT / relative]
    if source and not pd.isna(source):
        candidates.append(Path(source))
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    return ''

source_paths = frame.source_path if 'source_path' in frame else pd.Series('', index=frame.index)
frame['image_path'] = [resolve_image_path(rel, src) for rel, src in zip(frame.filepath, source_paths)]
if frame.image_path.eq('').any():
    raise FileNotFoundError(frame.loc[frame.image_path.eq(''), 'filepath'].iloc[0])

source_identity = {
    'source_manifest_sha256': sha256_file(SOURCE_MANIFEST),
    'rows': len(frame),
    'class_counts': frame.label.value_counts().reindex(CLASSES).to_dict(),
}
print(json.dumps(source_identity, indent=2))


## 3. Frozen SigLIP2 embeddings from four rotations

Each rotation is embedded independently and L2-normalized. The four normalized vectors are averaged and normalized again for rotation-invariant neighbour search. Per-rotation vectors remain available for stability analysis. The fixed-resolution SigLIP2 processor and the ConvNeXt input pipeline do not preserve geometry in exactly the same way, so their agreement is a cross-model diagnostic rather than a pure backbone comparison.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
dtype = torch.float16 if device.type in {'cuda', 'mps'} else torch.float32
runtime_embed_batch_size = min(EMBED_BATCH_SIZE, 8) if device.type == 'mps' else EMBED_BATCH_SIZE
processor = None
model = None

def ensure_embedding_model():
    global processor, model
    if model is None:
        processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
        model = AutoModel.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, dtype=dtype
        ).to(device).eval()
        print('Loaded SigLIP2:', device, '| dtype:', dtype, '| batch:', runtime_embed_batch_size)

TRANSPOSES = {
    0: None,
    90: Image.Transpose.ROTATE_90,
    180: Image.Transpose.ROTATE_180,
    270: Image.Transpose.ROTATE_270,
}

def load_rotated(path, angle):
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened).convert('RGB')
    return image if angle == 0 else image.transpose(TRANSPOSES[angle])

def unwrap_image_features(vision_output):
    if torch.is_tensor(vision_output):
        return vision_output
    if getattr(vision_output, 'pooler_output', None) is not None:
        return vision_output.pooler_output
    if isinstance(vision_output, tuple) and len(vision_output) > 1:
        return vision_output[1]
    raise TypeError(f'Unsupported image-feature output: {type(vision_output).__name__}')

@torch.inference_mode()
def embed_images(images):
    ensure_embedding_model()
    inputs = processor(images=images, return_tensors='pt')
    inputs = {
        key: value.to(device=device, dtype=dtype) if key == 'pixel_values'
        else value.to(device)
        for key, value in inputs.items()
    }
    vision_output = model.get_image_features(**inputs)
    features = unwrap_image_features(vision_output)
    return F.normalize(features.float(), dim=1).cpu().numpy()

rotation_embedding_path = OUTPUT_DIR / 'siglip2_four_rotation_embeddings.npy'
mean_embedding_path = OUTPUT_DIR / 'siglip2_mean_embeddings.npy'
previous_summary_path = OUTPUT_DIR / 'stage5_summary.json'
previous_summary = (
    json.loads(previous_summary_path.read_text())
    if previous_summary_path.is_file() else {}
)
embedding_cache_identity_matches = (
    previous_summary.get('source_manifest_sha256') == source_identity['source_manifest_sha256']
    and previous_summary.get('model_id') == MODEL_ID
    and previous_summary.get('model_revision') == MODEL_REVISION
)
if (
    rotation_embedding_path.is_file()
    and mean_embedding_path.is_file()
    and embedding_cache_identity_matches
):
    rotation_embeddings = np.load(rotation_embedding_path).astype(np.float32)
    mean_embeddings = np.load(mean_embedding_path).astype(np.float32)
    if rotation_embeddings.shape[:2] != (len(frame), len(ROTATIONS)):
        raise ValueError('Saved rotation embeddings do not align with the current manifest.')
    if mean_embeddings.shape[0] != len(frame):
        raise ValueError('Saved mean embeddings do not align with the current manifest.')
    print('Reused validated SigLIP2 embeddings without loading the model:', rotation_embedding_path)
else:
    rotation_chunks = {angle: [] for angle in ROTATIONS}
    for start in tqdm(range(0, len(frame), runtime_embed_batch_size), desc='SigLIP2 four rotations'):
        paths = frame.image_path.iloc[start:start + runtime_embed_batch_size]
        for angle in ROTATIONS:
            rotation_chunks[angle].append(embed_images([load_rotated(path, angle) for path in paths]))

    rotation_embeddings = np.stack([
        np.concatenate(rotation_chunks[angle], axis=0) for angle in ROTATIONS
    ], axis=1).astype(np.float32)
    mean_embeddings = rotation_embeddings.mean(axis=1)
    mean_embeddings /= np.maximum(np.linalg.norm(mean_embeddings, axis=1, keepdims=True), 1e-12)

    np.save(rotation_embedding_path, rotation_embeddings.astype(np.float16))
    np.save(mean_embedding_path, mean_embeddings.astype(np.float16))

rotation_to_mean_cosine = np.einsum('nrd,nd->nr', rotation_embeddings, mean_embeddings)
pd.DataFrame({
    'filepath': frame.filepath,
    'rotation_min_cosine_to_mean': rotation_to_mean_cosine.min(axis=1),
    'rotation_mean_cosine_to_mean': rotation_to_mean_cosine.mean(axis=1),
}).to_csv(OUTPUT_DIR / 'rotation_embedding_stability.csv', index=False)
print('embedding shape:', mean_embeddings.shape)


## 4. Calibrate embedding recall with synthetic duplicates

The candidate threshold is estimated from this dataset. A stratified sample is transformed with free-angle rotation, Gaussian noise, blur, and JPEG recompression. The lower-tail similarity of these known-positive pairs defines the candidate threshold and reports estimated synthetic recall. This calibration is diagnostic; it does not create graph edges.

In [ ]:
def make_synthetic_duplicate(path, rng):
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened).convert('RGB')
    angle = float(rng.uniform(-25, 25))
    image = image.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True, fillcolor=(127, 127, 127))
    array = np.asarray(image).astype(np.float32)
    sigma = float(rng.uniform(5, 30))
    array = np.clip(array + rng.normal(0, sigma, array.shape), 0, 255).astype(np.uint8)
    if rng.random() < 0.5:
        array = cv2.GaussianBlur(array, (5, 5), sigmaX=float(rng.uniform(0.4, 1.4)))
    encode_quality = int(rng.integers(50, 91))
    success, encoded = cv2.imencode('.jpg', cv2.cvtColor(array, cv2.COLOR_RGB2BGR),
                                    [cv2.IMWRITE_JPEG_QUALITY, encode_quality])
    if not success:
        raise RuntimeError('Synthetic JPEG encoding failed.')
    decoded = cv2.cvtColor(cv2.imdecode(encoded, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    return Image.fromarray(decoded)


calibration_report_path = OUTPUT_DIR / 'embedding_threshold_calibration.json'
calibration_rows_path = OUTPUT_DIR / 'synthetic_duplicate_embedding_calibration.csv'
if (
    calibration_report_path.is_file()
    and calibration_rows_path.is_file()
    and embedding_cache_identity_matches
):
    calibration_report = json.loads(calibration_report_path.read_text())
    calibration_rows = pd.read_csv(calibration_rows_path)
    required_calibration_columns = {'row_index', 'synthetic_positive_cosine'}
    if not required_calibration_columns.issubset(calibration_rows.columns):
        raise ValueError('Cached synthetic calibration is missing required columns.')
    calibration_indices = calibration_rows.row_index.to_numpy(dtype=int)
    synthetic_positive_cosine = calibration_rows.synthetic_positive_cosine.to_numpy(np.float32)
    embedding_candidate_cosine = float(calibration_report['embedding_candidate_cosine'])
    synthetic_recall_at_candidate = float(calibration_report['estimated_synthetic_recall'])
    print('Reused validated synthetic calibration without loading SigLIP2.')
else:
    rng = np.random.default_rng(CALIBRATION_SEED)
    calibration_indices = []
    per_class = max(1, CALIBRATION_SAMPLE_SIZE // len(CLASSES))
    for label in CLASSES:
        candidates = frame.index[frame.label.eq(label)].to_numpy()
        calibration_indices.extend(rng.choice(candidates, size=min(per_class, len(candidates)), replace=False))
    calibration_indices = np.asarray(sorted(set(calibration_indices)))

    synthetic_mean_chunks = []
    for start in tqdm(range(0, len(calibration_indices), runtime_embed_batch_size), desc='synthetic duplicate calibration'):
        indices = calibration_indices[start:start + runtime_embed_batch_size]
        synthetic_images = [make_synthetic_duplicate(frame.image_path.iloc[index], rng) for index in indices]
        rotated_features = []
        for angle in ROTATIONS:
            rotated = [image if angle == 0 else image.transpose(TRANSPOSES[angle]) for image in synthetic_images]
            rotated_features.append(embed_images(rotated))
        chunk = np.stack(rotated_features, axis=1).mean(axis=1)
        chunk /= np.maximum(np.linalg.norm(chunk, axis=1, keepdims=True), 1e-12)
        synthetic_mean_chunks.append(chunk)

    synthetic_mean_embeddings = np.concatenate(synthetic_mean_chunks)
    synthetic_positive_cosine = np.einsum(
        'nd,nd->n', mean_embeddings[calibration_indices], synthetic_mean_embeddings
    )
    embedding_candidate_cosine = float(np.quantile(synthetic_positive_cosine, SYNTHETIC_RECALL_QUANTILE))
    synthetic_recall_at_candidate = float((synthetic_positive_cosine >= embedding_candidate_cosine).mean())
    calibration_report = {
        'sample_rows': int(len(calibration_indices)),
        'candidate_threshold_quantile': SYNTHETIC_RECALL_QUANTILE,
        'embedding_candidate_cosine': embedding_candidate_cosine,
        'estimated_synthetic_recall': synthetic_recall_at_candidate,
        'cosine_q01': float(np.quantile(synthetic_positive_cosine, 0.01)),
        'cosine_q05': float(np.quantile(synthetic_positive_cosine, 0.05)),
        'cosine_median': float(np.median(synthetic_positive_cosine)),
    }
    pd.DataFrame({
        'row_index': calibration_indices,
        'filepath': frame.filepath.iloc[calibration_indices].to_numpy(),
        'synthetic_positive_cosine': synthetic_positive_cosine,
    }).to_csv(calibration_rows_path, index=False)
    calibration_report_path.write_text(json.dumps(calibration_report, indent=2) + '\n')
print(json.dumps(calibration_report, indent=2))


## 5. Incremental embedding neighbours and geometric diagnostics

SigLIP2 proposes candidates using the calibrated threshold. Validated embeddings and synthetic calibration are reused without loading SigLIP2. Existing geometry metrics are migrated from the previous 5,000-candidate run; only missing candidates are verified, with an atomic checkpoint every 250 pairs. A stopped run resumes from that checkpoint.

Geometry is measured after denoising and mild blur. The notebook reports verification rates by cosine bin before any component is built. Very-high-cosine mutual neighbours may be merged directly only when the empirical upper-bin geometry rate supports that threshold; the gray zone requires geometry.

In [ ]:
def chunked_topk_cosine(embeddings, k, block_size):
    bank = torch.from_numpy(embeddings).to(device)
    all_scores, all_indices = [], []
    for start in tqdm(range(0, len(bank), block_size), desc='cosine kNN'):
        scores = bank[start:start + block_size] @ bank.T
        local_rows = torch.arange(scores.shape[0], device=device)
        scores[local_rows, start + local_rows] = -1
        values, indices = scores.topk(k=min(k, len(bank) - 1), dim=1)
        all_scores.append(values.cpu().numpy())
        all_indices.append(indices.cpu().numpy())
    return np.concatenate(all_scores), np.concatenate(all_indices)


neighbour_scores, neighbour_indices = chunked_topk_cosine(
    mean_embeddings, K_NEIGHBOURS, KNN_BLOCK_SIZE
)
pair_scores = {}
for first, (scores, indices) in enumerate(zip(neighbour_scores, neighbour_indices)):
    for score, second in zip(scores, indices):
        if frame.pixel_sha256.iloc[first] == frame.pixel_sha256.iloc[int(second)]:
            continue
        key = tuple(sorted((first, int(second))))
        pair_scores[key] = max(pair_scores.get(key, -1), float(score))

ranked_pair_pool = sorted(pair_scores.items(), key=lambda item: (-item[1], item[0]))
if not ranked_pair_pool:
    raise RuntimeError('No non-exact embedding-neighbour pairs were found.')

budget_index = min(MAX_GEOMETRY_CANDIDATES, len(ranked_pair_pool)) - 1
budget_cosine = float(ranked_pair_pool[budget_index][1])
operational_candidate_cosine = max(embedding_candidate_cosine, budget_cosine)
threshold_pairs = [
    item for item in ranked_pair_pool if item[1] >= operational_candidate_cosine
]

# Keep every tie at the data-derived budget boundary; do not silently truncate
# an equivalent-similarity pair. A pathological tie explosion is a hard stop.
if len(threshold_pairs) > math.ceil(MAX_GEOMETRY_CANDIDATES * 1.05):
    raise RuntimeError(
        f'{len(threshold_pairs)} candidates including boundary ties exceed the '
        f'5% tolerance above MAX_GEOMETRY_CANDIDATES={MAX_GEOMETRY_CANDIDATES}.'
    )
candidate_pairs = threshold_pairs
candidate_selection_report = {
    'unique_knn_pair_pool': int(len(ranked_pair_pool)),
    'synthetic_recall_threshold': float(embedding_candidate_cosine),
    'budget_cosine_threshold': budget_cosine,
    'operational_candidate_cosine': float(operational_candidate_cosine),
    'pairs_above_operational_threshold': int(len(threshold_pairs)),
    'geometry_candidates_retained': int(len(candidate_pairs)),
    'boundary_ties_retained_above_budget': int(max(0, len(candidate_pairs) - MAX_GEOMETRY_CANDIDATES)),
    'estimated_synthetic_recall_at_operational_threshold': float(
        (synthetic_positive_cosine >= operational_candidate_cosine).mean()
    ),
}
(OUTPUT_DIR / 'candidate_selection_report.json').write_text(
    json.dumps(candidate_selection_report, indent=2) + '\n'
)
print(json.dumps(candidate_selection_report, indent=2))


In [ ]:
def load_gray(path, maximum_side=1000):
    with Image.open(path) as opened:
        rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    scale = min(1.0, maximum_side / max(gray.shape))
    return cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else gray

cv2.setNumThreads(1)
_geometry_local = threading.local()

def thread_sift():
    if not hasattr(_geometry_local, 'sift'):
        _geometry_local.sift = cv2.SIFT_create(nfeatures=4000)
    return _geometry_local.sift

def verify_pair(path_a, path_b):
    sift = thread_sift()
    raw_a, raw_b = load_gray(path_a), load_gray(path_b)
    image_a = cv2.GaussianBlur(cv2.fastNlMeansDenoising(raw_a, None, 10, 7, 21), (3, 3), 0)
    image_b = cv2.GaussianBlur(cv2.fastNlMeansDenoising(raw_b, None, 10, 7, 21), (3, 3), 0)
    key_a, desc_a = sift.detectAndCompute(image_a, None)
    key_b, desc_b = sift.detectAndCompute(image_b, None)
    good = []
    if desc_a is not None and desc_b is not None:
        for pair in cv2.BFMatcher(cv2.NORM_L2).knnMatch(desc_a, desc_b, k=2):
            if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance:
                good.append(pair[0])
    result = {'sift_good_matches': len(good), 'homography_inlier_ratio': 0.0,
              'aligned_gray_correlation': np.nan}
    if len(good) < 4:
        return result
    source = np.float32([key_b[item.trainIdx].pt for item in good]).reshape(-1, 1, 2)
    destination = np.float32([key_a[item.queryIdx].pt for item in good]).reshape(-1, 1, 2)
    transform, inliers = cv2.findHomography(source, destination, cv2.RANSAC, 4)
    if transform is None:
        return result
    warped = cv2.warpPerspective(image_b, transform, (image_a.shape[1], image_a.shape[0]))
    mask = cv2.warpPerspective(np.ones_like(image_b, dtype=np.uint8), transform,
                               (image_a.shape[1], image_a.shape[0])).astype(bool)
    result['homography_inlier_ratio'] = float(inliers.mean())
    if mask.sum() > 100:
        first, second = image_a[mask].astype(float), warped[mask].astype(float)
        if first.std() > 0 and second.std() > 0:
            result['aligned_gray_correlation'] = float(np.corrcoef(first, second)[0, 1])
    return result

top_nonexact = {}
for first, indices in enumerate(neighbour_indices):
    for second in indices:
        second = int(second)
        if frame.pixel_sha256.iloc[first] != frame.pixel_sha256.iloc[second]:
            top_nonexact[first] = second
            break

def verify_candidate(candidate):
    (first, second), similarity = candidate
    metrics = verify_pair(frame.image_path.iloc[first], frame.image_path.iloc[second])
    geometry_verified = (
        metrics['sift_good_matches'] >= SIFT_MIN_MATCHES
        and metrics['homography_inlier_ratio'] >= HOMOGRAPHY_MIN_INLIER
        and pd.notna(metrics['aligned_gray_correlation'])
        and metrics['aligned_gray_correlation'] >= ALIGNED_MIN_CORRELATION
    )
    return {
        'index_a': first, 'index_b': second,
        'filepath_a': frame.filepath.iloc[first], 'filepath_b': frame.filepath.iloc[second],
        'label_a': frame.label.iloc[first], 'label_b': frame.label.iloc[second],
        'embedding_cosine': similarity,
        'mutual_top1_nonexact': top_nonexact.get(first) == second and top_nonexact.get(second) == first,
        **metrics, 'geometry_verified': geometry_verified,
    }

# SigLIP2 is no longer needed; free unified memory before CPU-heavy geometry.
del model, processor
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()
elif device.type == 'mps':
    torch.mps.empty_cache()

print('Geometry workers:', GEOMETRY_WORKERS)
edge_columns = [
    'index_a', 'index_b', 'filepath_a', 'filepath_b', 'label_a', 'label_b',
    'embedding_cosine', 'mutual_top1_nonexact', 'sift_good_matches',
    'homography_inlier_ratio', 'aligned_gray_correlation', 'geometry_verified',
]
candidate_keys = {pair for pair, _ in candidate_pairs}
geometry_cache_path = OUTPUT_DIR / 'geometry_metrics_cache.csv'
geometry_cache_meta_path = OUTPUT_DIR / 'geometry_metrics_cache_meta.json'
legacy_geometry_path = OUTPUT_DIR / 'embedding_duplicate_candidates.csv'
cached_by_key = {}

# Migrate the already-computed 5,000 rows, then prefer newer checkpoint rows.
for cache_path in (legacy_geometry_path, geometry_cache_path):
    if not cache_path.is_file():
        continue
    cached = pd.read_csv(cache_path)
    if not set(edge_columns).issubset(cached.columns):
        continue
    for row in cached[edge_columns].itertuples(index=False):
        key = (int(row.index_a), int(row.index_b))
        if key not in candidate_keys:
            continue
        if (
            row.filepath_a != frame.filepath.iloc[key[0]]
            or row.filepath_b != frame.filepath.iloc[key[1]]
        ):
            raise RuntimeError(f'Geometry cache filepath mismatch for pair {key}.')
        cached_by_key[key] = row._asdict()

edge_by_key = {}
for candidate in candidate_pairs:
    key, similarity = candidate
    if key not in cached_by_key:
        continue
    cached = cached_by_key[key]
    metrics = {
        'sift_good_matches': int(cached['sift_good_matches']),
        'homography_inlier_ratio': float(cached['homography_inlier_ratio']),
        'aligned_gray_correlation': float(cached['aligned_gray_correlation']),
    }
    geometry_verified = (
        metrics['sift_good_matches'] >= SIFT_MIN_MATCHES
        and metrics['homography_inlier_ratio'] >= HOMOGRAPHY_MIN_INLIER
        and pd.notna(metrics['aligned_gray_correlation'])
        and metrics['aligned_gray_correlation'] >= ALIGNED_MIN_CORRELATION
    )
    first, second = key
    edge_by_key[key] = {
        'index_a': first, 'index_b': second,
        'filepath_a': frame.filepath.iloc[first], 'filepath_b': frame.filepath.iloc[second],
        'label_a': frame.label.iloc[first], 'label_b': frame.label.iloc[second],
        'embedding_cosine': similarity,
        'mutual_top1_nonexact': top_nonexact.get(first) == second and top_nonexact.get(second) == first,
        **metrics, 'geometry_verified': geometry_verified,
    }

def save_geometry_checkpoint():
    ordered_rows = [edge_by_key[pair] for pair, _ in candidate_pairs if pair in edge_by_key]
    checkpoint = pd.DataFrame(ordered_rows, columns=edge_columns)
    temporary = geometry_cache_path.with_suffix('.csv.tmp')
    checkpoint.to_csv(temporary, index=False)
    temporary.replace(geometry_cache_path)
    geometry_cache_meta_path.write_text(json.dumps({
        'source_manifest_sha256': source_identity['source_manifest_sha256'],
        'verifier_version': GEOMETRY_VERIFIER_VERSION,
        'rows_completed': len(checkpoint),
        'target_rows': len(candidate_pairs),
    }, indent=2) + '\n')

missing_candidates = [candidate for candidate in candidate_pairs if candidate[0] not in edge_by_key]
print(
    f'Reused {len(edge_by_key):,} geometry rows; '
    f'{len(missing_candidates):,} of {len(candidate_pairs):,} remain.'
)
if edge_by_key:
    save_geometry_checkpoint()
if missing_candidates:
    with ThreadPoolExecutor(max_workers=GEOMETRY_WORKERS) as executor:
        for start in range(0, len(missing_candidates), GEOMETRY_CHECKPOINT_EVERY):
            batch = missing_candidates[start:start + GEOMETRY_CHECKPOINT_EVERY]
            rows = list(tqdm(
                executor.map(verify_candidate, batch), total=len(batch),
                desc=(
                    f'geometry {len(edge_by_key):,}/'
                    f'{len(candidate_pairs):,}'
                ),
            ))
            for row in rows:
                edge_by_key[(int(row['index_a']), int(row['index_b']))] = row
            save_geometry_checkpoint()

edge_rows = [edge_by_key[pair] for pair, _ in candidate_pairs]
edges = pd.DataFrame(edge_rows, columns=edge_columns)

if len(edges):
    lower = np.floor(edges.embedding_cosine.min() * 100) / 100
    bin_edges = np.unique(np.append(np.arange(lower, 1.001, 0.01), 1.000001))
    edges['cosine_bin'] = pd.cut(edges.embedding_cosine, bin_edges, include_lowest=True)
    cosine_diagnostics = edges.groupby('cosine_bin', observed=True).agg(
        pairs=('index_a', 'size'),
        geometry_verified_rate=('geometry_verified', 'mean'),
        median_matches=('sift_good_matches', 'median'),
        median_inlier=('homography_inlier_ratio', 'median'),
        median_correlation=('aligned_gray_correlation', 'median'),
    ).reset_index()
    cosine_diagnostics['cosine_bin'] = cosine_diagnostics.cosine_bin.astype(str)
else:
    cosine_diagnostics = pd.DataFrame(columns=[
        'cosine_bin', 'pairs', 'geometry_verified_rate', 'median_matches',
        'median_inlier', 'median_correlation'
    ])
cosine_diagnostics.to_csv(OUTPUT_DIR / 'cosine_geometry_diagnostics.csv', index=False)

direct_merge_threshold = None
if len(edges):
    candidate_lowers = sorted({float(interval.left) for interval in edges.cosine_bin.dropna()}, reverse=True)
    eligible_thresholds = []
    for threshold in candidate_lowers:
        upper = edges.embedding_cosine >= threshold
        if upper.sum() >= DIRECT_MERGE_MIN_PAIRS and edges.loc[upper, 'geometry_verified'].mean() >= DIRECT_MERGE_MIN_GEOMETRY_RATE:
            eligible_thresholds.append(threshold)
    if eligible_thresholds:
        direct_merge_threshold = min(eligible_thresholds)

edges['direct_high_cosine_edge'] = False
if direct_merge_threshold is not None:
    edges['direct_high_cosine_edge'] = (
        edges.mutual_top1_nonexact & edges.embedding_cosine.ge(direct_merge_threshold)
    )
edges['merge_edge'] = edges.geometry_verified | edges.direct_high_cosine_edge
edges['merge_reason'] = np.select(
    [edges.geometry_verified & edges.direct_high_cosine_edge,
     edges.direct_high_cosine_edge, edges.geometry_verified],
    ['geometry_and_direct_cosine', 'direct_cosine', 'geometry'], default='rejected'
)
edges.to_csv(OUTPUT_DIR / 'embedding_duplicate_candidates.csv', index=False)
edges.loc[edges.merge_edge].to_csv(OUTPUT_DIR / 'accepted_duplicate_edges.csv', index=False)
print(cosine_diagnostics.to_string(index=False))
print('data-derived direct threshold:', direct_merge_threshold)
print('accepted edges:', int(edges.merge_edge.sum()),
      '| cross-label:', int((edges.merge_edge & edges.label_a.ne(edges.label_b)).sum()))


## 6. Components, unseen label-noisy holdout, and new grouped folds

Components are rebuilt from scratch before any partition is selected. Exact pixel identity and verified embedding edges are unioned. Mixed-label components remain intact and are reported for review. The component-level holdout split is then audited by SigLIP2 nearest-neighbour similarity against development; this diagnoses residual cross-partition similarity without claiming that cosine alone proves duplication.

In [ ]:
parent = np.arange(len(frame))

def find(item):
    while parent[item] != item:
        parent[item] = parent[parent[item]]
        item = parent[item]
    return int(item)

def union(first, second):
    root_a, root_b = find(int(first)), find(int(second))
    if root_a != root_b:
        parent[root_b] = root_a

for indices in frame.groupby('pixel_sha256').indices.values():
    indices = list(indices)
    for item in indices[1:]:
        union(indices[0], item)
for row in edges.loc[edges.merge_edge].itertuples(index=False):
    union(row.index_a, row.index_b)

components = {}
for index in range(len(frame)):
    components.setdefault(find(index), []).append(index)
ordered_components = sorted(components.values(), key=lambda values: (-len(values), frame.filepath.iloc[values].min()))
index_to_group = {}
for values in ordered_components:
    identity = '\n'.join(sorted(frame.filepath.iloc[values])).encode()
    group_id = 'sigdup_' + hashlib.sha1(identity).hexdigest()[:12]
    for index in values:
        index_to_group[index] = group_id
frame['dedup_group'] = [index_to_group[index] for index in range(len(frame))]

component_summary = frame.groupby('dedup_group').agg(
    component_size=('filepath', 'size'), label_count=('label', 'nunique'),
    labels=('label', lambda values: '|'.join(sorted(set(values)))),
).reset_index()
component_summary.to_csv(OUTPUT_DIR / 'duplicate_components.csv', index=False)
component_histogram = component_summary.component_size.value_counts().sort_index().rename_axis(
    'component_size'
).reset_index(name='component_count')
component_histogram.to_csv(OUTPUT_DIR / 'component_size_histogram.csv', index=False)
mixed_component_mask = component_summary.label_count.gt(1)
component_size_report = {
    'components': int(len(component_summary)),
    'mixed_label_components': int(mixed_component_mask.sum()),
    'rows_in_mixed_label_components': int(component_summary.loc[mixed_component_mask, 'component_size'].sum()),
    'component_size_p50': float(component_summary.component_size.quantile(0.50)),
    'component_size_p90': float(component_summary.component_size.quantile(0.90)),
    'component_size_p95': float(component_summary.component_size.quantile(0.95)),
    'component_size_p99': float(component_summary.component_size.quantile(0.99)),
    'component_size_max': int(component_summary.component_size.max()),
}
(OUTPUT_DIR / 'component_size_report.json').write_text(
    json.dumps(component_size_report, indent=2) + '\n'
)
diagnostic_component_limit = max(25, int(np.ceil(0.005 * len(frame))))
hard_stop_component_limit = max(250, int(np.ceil(0.02 * len(frame))))
large_components = component_summary.loc[
    component_summary.component_size.gt(diagnostic_component_limit)
].copy()

accepted_with_group = edges.loc[edges.merge_edge].copy()
accepted_with_group['dedup_group'] = frame.dedup_group.iloc[
    accepted_with_group.index_a.to_numpy(dtype=int)
].to_numpy()
accepted_with_group['cross_label'] = accepted_with_group.label_a.ne(
    accepted_with_group.label_b
)
large_edge_diagnostics = accepted_with_group.groupby('dedup_group').agg(
    accepted_edges=('merge_edge', 'size'),
    geometry_verified_rate=('geometry_verified', 'mean'),
    cross_label_edges=('cross_label', 'sum'),
    embedding_cosine_min=('embedding_cosine', 'min'),
    embedding_cosine_median=('embedding_cosine', 'median'),
    homography_inlier_median=('homography_inlier_ratio', 'median'),
    aligned_correlation_median=('aligned_gray_correlation', 'median'),
).reset_index()
large_component_diagnostics = large_components.merge(
    large_edge_diagnostics, on='dedup_group', how='left'
).sort_values('component_size', ascending=False)
large_component_diagnostics.to_csv(
    OUTPUT_DIR / 'large_component_diagnostics.csv', index=False
)
if len(large_component_diagnostics):
    print(
        f'Diagnostic only: {len(large_component_diagnostics)} component(s) exceed '
        f'{diagnostic_component_limit} rows. See large_component_diagnostics.csv.'
    )
if component_size_report['component_size_max'] > hard_stop_component_limit:
    raise RuntimeError(
        f"Largest component has {component_size_report['component_size_max']} rows, above "
        f"the hard-stop limit {hard_stop_component_limit}. Inspect stock-style false merges before splitting."
    )
print(json.dumps(component_size_report, indent=2))

y = frame.label.map(CLASS_TO_INDEX).to_numpy()
groups = frame.dedup_group.to_numpy()
holdout_splitter = StratifiedGroupKFold(
    n_splits=HOLDOUT_FRACTION_FOLDS, shuffle=True, random_state=HOLDOUT_SEED
)
development_indices, holdout_indices = next(holdout_splitter.split(frame, y, groups))
frame['partition'] = 'development'
frame.loc[holdout_indices, 'partition'] = 'holdout'
frame['fold'] = -1

development = frame.index[frame.partition.eq('development')].to_numpy()
fold_splitter = StratifiedGroupKFold(n_splits=OOF_FOLDS, shuffle=True, random_state=FOLD_SEED)
for fold, (_, valid_local) in enumerate(fold_splitter.split(
    frame.loc[development], y[development], groups[development]
)):
    frame.loc[development[valid_local], 'fold'] = fold

assert frame.loc[frame.partition.eq('development'), 'fold'].ge(0).all()
assert frame.loc[frame.partition.eq('holdout'), 'fold'].eq(-1).all()
assert frame.groupby('dedup_group').partition.nunique().max() == 1
assert frame.loc[frame.partition.eq('development')].groupby('dedup_group').fold.nunique().max() == 1

# Cross-partition nearest-neighbour audit. This is diagnostic: high semantic cosine
# is not automatically a duplicate, but a high tail must be reviewed before the
# holdout is treated as trustworthy.
development = frame.index[frame.partition.eq('development')].to_numpy()
holdout = frame.index[frame.partition.eq('holdout')].to_numpy()
development_embeddings = np.ascontiguousarray(mean_embeddings[development], dtype=np.float32)
holdout_embeddings = np.ascontiguousarray(mean_embeddings[holdout], dtype=np.float32)
nearest_cosine = np.empty(len(holdout), dtype=np.float32)
nearest_development_local = np.empty(len(holdout), dtype=np.int64)
for start in range(0, len(holdout), 64):
    similarities = holdout_embeddings[start:start + 64] @ development_embeddings.T
    nearest = similarities.argmax(axis=1)
    nearest_cosine[start:start + len(nearest)] = similarities[np.arange(len(nearest)), nearest]
    nearest_development_local[start:start + len(nearest)] = nearest
nearest_development = development[nearest_development_local]

edge_lookup = {
    (int(row.index_a), int(row.index_b)): row
    for row in edges.itertuples(index=False)
}
holdout_nn_rows = []
for holdout_index, development_index, cosine in zip(
    holdout, nearest_development, nearest_cosine
):
    key = tuple(sorted((int(holdout_index), int(development_index))))
    edge = edge_lookup.get(key)
    holdout_nn_rows.append({
        'holdout_index': int(holdout_index),
        'holdout_filepath': frame.filepath.iloc[holdout_index],
        'holdout_label_original': frame.label.iloc[holdout_index],
        'holdout_group': frame.dedup_group.iloc[holdout_index],
        'nearest_development_index': int(development_index),
        'nearest_development_filepath': frame.filepath.iloc[development_index],
        'nearest_development_label_original': frame.label.iloc[development_index],
        'nearest_development_group': frame.dedup_group.iloc[development_index],
        'nearest_development_cosine': float(cosine),
        'candidate_geometry_available': edge is not None,
        'geometry_verified': bool(edge.geometry_verified) if edge is not None else False,
        'merge_edge': bool(edge.merge_edge) if edge is not None else False,
    })
holdout_nn = pd.DataFrame(holdout_nn_rows).sort_values(
    'nearest_development_cosine', ascending=False
)
holdout_nn.to_csv(OUTPUT_DIR / 'holdout_to_development_nearest_neighbors.csv', index=False)
if holdout_nn.merge_edge.any():
    raise RuntimeError('Accepted duplicate edge crosses development/holdout after grouping.')
holdout_nn_report = {
    'status': 'unseen_from_model_selection_but_original_labels_may_be_noisy',
    'holdout_rows': int(len(holdout_nn)),
    'review_cosine': HOLDOUT_NN_REVIEW_COSINE,
    'cosine_quantiles': {
        str(quantile): float(holdout_nn.nearest_development_cosine.quantile(quantile))
        for quantile in (0.0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0)
    },
    'counts_at_or_above': {
        str(threshold): int(holdout_nn.nearest_development_cosine.ge(threshold).sum())
        for threshold in HOLDOUT_NN_THRESHOLDS
    },
    'rates_at_or_above': {
        str(threshold): float(holdout_nn.nearest_development_cosine.ge(threshold).mean())
        for threshold in HOLDOUT_NN_THRESHOLDS
    },
    'cross_label_at_or_above_review_cosine': int((
        holdout_nn.nearest_development_cosine.ge(HOLDOUT_NN_REVIEW_COSINE)
        & holdout_nn.holdout_label_original.ne(
            holdout_nn.nearest_development_label_original
        )
    ).sum()),
    'accepted_duplicate_edges_crossing_partition': int(holdout_nn.merge_edge.sum()),
}
(OUTPUT_DIR / 'holdout_nearest_neighbor_report.json').write_text(
    json.dumps(holdout_nn_report, indent=2) + '\n'
)
print(json.dumps(holdout_nn_report, indent=2))

manifest_columns = [
    'filepath', 'filename', 'label', 'pixel_sha256', 'dedup_group', 'partition', 'fold', 'image_path'
]
frame[manifest_columns].to_csv(OUTPUT_DIR / 'siglip2_grouped_manifest.csv', index=False)
frame.loc[frame.partition.eq('development'), manifest_columns].to_csv(
    OUTPUT_DIR / 'convnext_development_manifest.csv', index=False
)
frame.loc[frame.partition.eq('holdout'), manifest_columns].to_csv(
    OUTPUT_DIR / 'frozen_holdout_manifest.csv', index=False
)
frame.loc[frame.partition.eq('holdout'), manifest_columns].to_csv(
    OUTPUT_DIR / 'unseen_label_noisy_holdout_manifest.csv', index=False
)
print(pd.crosstab([frame.partition, frame.fold], frame.label).reindex(columns=CLASSES, fill_value=0))
holdout_support = frame.loc[frame.partition.eq('holdout'), 'label'].value_counts().reindex(CLASSES, fill_value=0)
if holdout_support.min() < 100:
    print('Holdout warning: per-class metrics will be high-variance; support =', holdout_support.to_dict())


## 7. Repeated grouped-CV SigLIP2 linear-probe OOF

Only development rows participate. Stability comes from repeating `StratifiedGroupKFold` with different fold seeds, so each row is predicted from materially different training partitions. Model initialization is held fixed within a fold. Training uses a fixed epoch count and cosine schedule; validation labels never select a checkpoint. Probe features concatenate the orientation-preserving 0° embedding with the four-rotation mean. The unseen holdout labels are not evaluated.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

class LinearProbe(nn.Module):
    def __init__(self, dimension, classes):
        super().__init__()
        self.head = nn.Linear(dimension, classes)
    def forward(self, inputs):
        return self.head(inputs)

def fit_probe(train_x, train_y, valid_x, seed):
    set_seed(seed)
    probe = LinearProbe(train_x.shape[1], len(CLASSES)).to(device)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=3e-3, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=LINEAR_PROBE_EPOCHS
    )
    dataset = TensorDataset(torch.from_numpy(train_x).float(), torch.from_numpy(train_y).long())
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(dataset, batch_size=LINEAR_PROBE_BATCH_SIZE, shuffle=True, generator=generator)
    for _ in range(LINEAR_PROBE_EPOCHS):
        probe.train()
        for batch_x, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(probe(batch_x.to(device)), batch_y.to(device), label_smoothing=0.02)
            loss.backward(); optimizer.step()
        scheduler.step()
    probe.eval()
    with torch.no_grad():
        valid_tensor = torch.from_numpy(valid_x).float().to(device)
        return probe(valid_tensor).softmax(1).cpu().numpy()

development_mask = frame.partition.eq('development').to_numpy()
development_rows = frame.loc[development_mask].copy()
if PROBE_FEATURE_MODE == 'rot0':
    all_probe_features = rotation_embeddings[:, 0]
elif PROBE_FEATURE_MODE == 'rot0_plus_mean':
    all_probe_features = np.concatenate([rotation_embeddings[:, 0], mean_embeddings], axis=1)
else:
    raise ValueError(f'Unknown PROBE_FEATURE_MODE: {PROBE_FEATURE_MODE}')
features = all_probe_features[development_mask].astype(np.float32)
targets = development_rows.label.map(CLASS_TO_INDEX).to_numpy()
development_groups = development_rows.dedup_group.to_numpy()

fold_seed_oof = {}
metric_rows = []
fold_assignment = development_rows[['filepath']].copy()
for fold_seed in PROBE_FOLD_SEEDS:
    splitter = StratifiedGroupKFold(n_splits=OOF_FOLDS, shuffle=True, random_state=fold_seed)
    folds_for_seed = np.full(len(development_rows), -1, dtype=np.int16)
    for fold, (_, valid_indices) in enumerate(splitter.split(features, targets, development_groups)):
        folds_for_seed[valid_indices] = fold
    assert (folds_for_seed >= 0).all()
    assert pd.DataFrame({'group': development_groups, 'fold': folds_for_seed}).groupby('group').fold.nunique().max() == 1
    fold_assignment[f'fold_seed_{fold_seed}'] = folds_for_seed

    oof = np.full((len(development_rows), len(CLASSES)), np.nan, dtype=np.float32)
    for fold in range(OOF_FOLDS):
        train_mask, valid_mask = folds_for_seed != fold, folds_for_seed == fold
        train_x, valid_x = features[train_mask], features[valid_mask]
        mean, std = train_x.mean(0), np.maximum(train_x.std(0), 1e-6)
        probabilities = fit_probe(
            (train_x - mean) / std, targets[train_mask], (valid_x - mean) / std,
            PROBE_MODEL_SEED + fold,
        )
        oof[valid_mask] = probabilities
        fold_macro_f1 = f1_score(
            targets[valid_mask], probabilities.argmax(1), average='macro', zero_division=0
        )
        metric_rows.append({
            'fold_seed': fold_seed, 'fold': fold, 'macro_f1_after_fixed_epochs': fold_macro_f1
        })
    assert np.isfinite(oof).all()
    fold_seed_oof[fold_seed] = oof
    np.save(OUTPUT_DIR / f'siglip2_linear_probe_oof_fold_seed_{fold_seed}.npy', oof)

probe_oof = np.mean(np.stack(list(fold_seed_oof.values())), axis=0)
np.save(OUTPUT_DIR / 'siglip2_linear_probe_oof_ensemble.npy', probe_oof)
pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR / 'siglip2_linear_probe_fold_metrics.csv', index=False)
fold_assignment.to_csv(OUTPUT_DIR / 'siglip2_probe_fold_assignments.csv', index=False)
pd.DataFrame({
    'filepath': development_rows.filepath,
    'label': development_rows.label,
    'primary_fold': development_rows.fold,
    **{f'p_{label}': probe_oof[:, index] for index, label in enumerate(CLASSES)},
}).to_csv(OUTPUT_DIR / 'siglip2_linear_probe_oof.csv', index=False)


## 8. Partition-stability agreement and new noise matrices

ConvNeXt OOF must use `convnext_development_manifest.csv`, the same row identities, and one fixed seed. Expected CSV columns are `filepath,p_Aman,p_Rusak,p_Lainnya`. Until that file exists, agreement measures only repeated grouped-CV partitions of the linear probe.

In [ ]:
def prediction_from_probabilities(probabilities):
    return np.asarray(CLASSES)[probabilities.argmax(1)]

def metric_table(probabilities, name):
    predicted = probabilities.argmax(1)
    precision, recall, f1, support = precision_recall_fscore_support(
        targets, predicted, labels=range(len(CLASSES)), zero_division=0
    )
    return pd.DataFrame({'model': name, 'label': CLASSES, 'precision': precision,
                         'recall': recall, 'f1': f1, 'support': support})

def save_noise_matrices(probabilities, prefix):
    predicted = probabilities.argmax(1)
    hard = confusion_matrix(targets, predicted, labels=range(len(CLASSES)))
    soft = np.stack([probabilities[targets == index].sum(0) for index in range(len(CLASSES))])
    for suffix, matrix in [('hard_counts', hard), ('hard_row_normalized', hard / hard.sum(1, keepdims=True)),
                           ('soft_row_normalized', soft / soft.sum(1, keepdims=True))]:
        pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).rename_axis('observed_label').to_csv(
            OUTPUT_DIR / f'{prefix}_noise_{suffix}.csv'
        )

agreement_rows = []
fold_seed_items = list(fold_seed_oof.items())
for first_index, (fold_seed_a, probability_a) in enumerate(fold_seed_items):
    for fold_seed_b, probability_b in fold_seed_items[first_index + 1:]:
        agreement_rows.append({
            'model_a': f'linear_grouped_cv_seed_{fold_seed_a}',
            'model_b': f'linear_grouped_cv_seed_{fold_seed_b}',
            'agreement_rate': float((probability_a.argmax(1) == probability_b.argmax(1)).mean()),
        })

partition_predictions = np.stack([probability.argmax(1) for _, probability in fold_seed_items])
partition_consensus = np.apply_along_axis(
    lambda values: np.bincount(values, minlength=len(CLASSES)).max() / len(values),
    axis=0, arr=partition_predictions,
)
partition_stability = pd.DataFrame({
    'filepath': development_rows.filepath,
    'label': development_rows.label,
    'partition_consensus_rate': partition_consensus,
    'partition_unanimous': partition_consensus == 1.0,
    'ensemble_prediction': prediction_from_probabilities(probe_oof),
})
partition_stability.to_csv(OUTPUT_DIR / 'siglip2_probe_partition_stability.csv', index=False)

metrics = [metric_table(probe_oof, 'siglip2_linear_probe_ensemble')]
save_noise_matrices(probe_oof, 'siglip2_linear_probe')

def load_aligned_oof(path):
    data = pd.read_csv(path)
    required = {'filepath', *(f'p_{label}' for label in CLASSES)}
    if not required.issubset(data.columns) or data.filepath.duplicated().any():
        raise ValueError(f'Invalid OOF contract: {path}')
    aligned = development_rows[['filepath']].merge(data, on='filepath', how='left', validate='one_to_one')
    if aligned.isna().any().any():
        raise ValueError(f'OOF rows do not align: {path}')
    return aligned[[f'p_{label}' for label in CLASSES]].to_numpy(float)

convnext_new = None
if CONVNEXT_NEW_GROUPS_OOF is not None:
    convnext_new = load_aligned_oof(CONVNEXT_NEW_GROUPS_OOF)
    metrics.append(metric_table(convnext_new, 'convnext_new_groups_one_seed'))
    save_noise_matrices(convnext_new, 'convnext_new_groups')
    agreement_rows.append({
        'model_a': 'convnext_new_groups_one_seed', 'model_b': 'siglip2_linear_probe_ensemble',
        'agreement_rate': float((convnext_new.argmax(1) == probe_oof.argmax(1)).mean()),
    })

pd.concat(metrics, ignore_index=True).to_csv(OUTPUT_DIR / 'oof_class_metrics.csv', index=False)
pd.DataFrame(agreement_rows).to_csv(OUTPUT_DIR / 'model_and_seed_agreement.csv', index=False)


## 9. Dedup delta and stage gate

Dedup delta is accepted only when old-group and new-group ConvNeXt OOF files come from the same architecture, seed, preprocessing, and hyperparameters. This cell refuses to infer a delta from the old ResNet OOF.

After the two files are supplied, inspect per-class changes and paired sample correctness. Bias tuning, noise filtering, and two-stage training remain downstream and must use development OOF only.

The unseen, label-noisy holdout is a **paired go/no-go check**, not a recipe-ranking set:

- lock one baseline and one final candidate before opening it;
- evaluate both on the same rows with cluster bootstrap by rebuilt duplicate component;
- treat small deltas as inconclusive, especially with few `Lainnya` rows;
- prove the frozen `Lainnya` decision-mask hash on development OOF before holdout evaluation;
- use the public leaderboard at most three times: baseline sanity, final candidate, and one contingency reserved for a verified pipeline bug.

After the recipe and epoch count are locked, retrain the submission model on all 12,224 labeled rows.

In [ ]:
stage5 = {
    'grouping_protocol_version': GROUPING_PROTOCOL_VERSION,
    'source_manifest_sha256': source_identity['source_manifest_sha256'],
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'rotations': list(ROTATIONS), 'rows': len(frame),
    'synthetic_recall_target_cosine': embedding_candidate_cosine,
    'estimated_synthetic_recall_at_target': synthetic_recall_at_candidate,
    'operational_candidate_cosine': candidate_selection_report['operational_candidate_cosine'],
    'geometry_candidates_retained': candidate_selection_report['geometry_candidates_retained'],
    'estimated_synthetic_recall_at_operational_threshold': (
        candidate_selection_report['estimated_synthetic_recall_at_operational_threshold']
    ),
    'data_derived_direct_merge_threshold': direct_merge_threshold,
    'accepted_duplicate_edges': int(edges.merge_edge.sum()),
    'dedup_components': int(frame.dedup_group.nunique()),
    'mixed_label_components': int(component_summary.label_count.gt(1).sum()),
    'rows_in_mixed_label_components': int(
        component_summary.loc[component_summary.label_count.gt(1), 'component_size'].sum()
    ),
    'component_size_p95': component_size_report['component_size_p95'],
    'component_size_p99': component_size_report['component_size_p99'],
    'component_size_max': component_size_report['component_size_max'],
    'development_rows': int(frame.partition.eq('development').sum()),
    'holdout_rows': int(frame.partition.eq('holdout').sum()),
    'holdout_class_support': holdout_support.to_dict(),
    'holdout_status': holdout_nn_report['status'],
    'holdout_nearest_neighbor_report': holdout_nn_report,
    'probe_feature_mode': PROBE_FEATURE_MODE,
    'probe_grouped_cv_seeds': list(PROBE_FOLD_SEEDS),
    'probe_partition_unanimous_rate': float(partition_stability.partition_unanimous.mean()),
    'convnext_new_oof_available': convnext_new is not None,
    'holdout_evaluated': bool(EVALUATE_HOLDOUT),
    'holdout_evaluation_role': 'paired_go_no_go_only_not_recipe_ranking',
    'public_leaderboard_submission_budget': PUBLIC_LEADERBOARD_SUBMISSION_BUDGET,
}

if CONVNEXT_OLD_GROUPS_OOF is not None and CONVNEXT_NEW_GROUPS_OOF is not None:
    old = load_aligned_oof(CONVNEXT_OLD_GROUPS_OOF)
    new = load_aligned_oof(CONVNEXT_NEW_GROUPS_OOF)
    old_correct = old.argmax(1) == targets
    new_correct = new.argmax(1) == targets
    delta = {
        'old_macro_f1': float(f1_score(targets, old.argmax(1), average='macro')),
        'new_macro_f1': float(f1_score(targets, new.argmax(1), average='macro')),
        'paired_gain_wrong_to_right': int((~old_correct & new_correct).sum()),
        'paired_harm_right_to_wrong': int((old_correct & ~new_correct).sum()),
    }
    stage5['dedup_delta'] = delta
else:
    stage5['dedup_delta'] = 'pending matched ConvNeXt old/new-group OOF files'

(OUTPUT_DIR / 'stage5_summary.json').write_text(json.dumps(stage5, indent=2) + '\n')
print(json.dumps(stage5, indent=2))

if EVALUATE_HOLDOUT:
    raise RuntimeError(
        'Holdout evaluation is intentionally not implemented in this exploratory notebook. '
        'Evaluate it once, from the final locked pipeline.'
    )
